# Frozen NEW Rater + Waterbirds-ERM Embeddings — Final Classifier

This notebook preserves the **same frozen-Rater final-classifier structure**
as the attached sandbox, but replaces:

- the old ImageNet-only embeddings with the persistent **2048-D embeddings
  from the Waterbirds-finetuned ERM ResNet-50**;
- the old Rater checkpoint with the **new `final_rater.pt`** from the
  500-step meta-learning experiment.

There is **no Rater training** here. The Rater is loaded, switched to
`eval()` mode, and completely frozen.

## Active hyperparameters preserved from the uploaded notebook

```python
SPLIT_VAL = 0.6
BATCH_SIZE = 128
FINAL_EPOCHS = 50

FINAL_LR = 1e-2
FINAL_MOMENTUM = 0.9
FINAL_WEIGHT_DECAY = 1e-4

NORMALIZATION_MODE = "global"
WEIGHT_TRANSFORM = "sigmoid"
WEIGHT_SCALE = 1.0
CORRECT_WEIGHT = 1.0
TEMPERATURE = 1.0

AUX_CE_COEFF = 0.0
SELECTION_METRIC = "wga"
PLOT_EVERY = 1
```

The attached notebook's **active code** used `NORMALIZATION_MODE="global"`
even though some markdown referred to the older `wrong_batch` baseline.
This notebook preserves the active value.

For each calibration minibatch:

1. run the current final classifier;
2. find the currently misclassified examples;
3. read the frozen-Rater raw score for only those wrong examples;
4. standardize using the fixed calibration-wide Rater mean/std;
5. apply sigmoid;
6. keep `weight=1` for correct examples;
7. use the Rater-derived weight for wrong examples;
8. optimize

\[
L_{\mathrm{weighted}}=\sum_i w_i CE_i
\]

with **no division by** `sum(weights)`.

The final classifier remains a **fresh random `Linear(2048,2)`**, as in the
attached notebook.

If `split_val=0.6` subset tensors are not already in the new ERM embedding
cache, this notebook derives them directly from the saved full `val.pt`
embedding tensor. It does not rerun ResNet-50.

In [ ]:
# ============================================================
# 0. Mount Drive, imports, paths, and EXPERIMENT HYPERPARAMETERS
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
import os
import json
import random
import csv

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


# ============================================================
# Paths
# ============================================================

DRIVE_ROOT = Path(
    '/content/drive/MyDrive/evidential_alignment_colab'
)

# Optional manual overrides.
EMBEDDING_CACHE_OVERRIDE = None
RATER_CHECKPOINT_OVERRIDE = None

EMBEDDING_CACHE_PARENT = (
    DRIVE_ROOT
    / 'embeddings_waterbirds_erm'
)


def _valid_erm_embedding_root(root):
    root = Path(root)

    required = [
        root / 'train_no_aug.pt',
        root / 'val.pt',
        root / 'test.pt',
    ]

    if not all(path.exists() for path in required):
        return False

    try:
        payload = torch.load(
            root / 'train_no_aug.pt',
            map_location='cpu',
            weights_only=False,
        )

        z = payload.get('embeddings')

        if (
            not torch.is_tensor(z)
            or z.ndim != 2
            or z.shape[1] != 2048
        ):
            return False

        metadata = payload.get(
            'cache_metadata',
            {},
        )

        representation = metadata.get(
            'representation_source'
        )

        if (
            representation
            and representation != 'waterbirds_erm_checkpoint'
        ):
            return False

        return True

    except Exception:
        return False


if EMBEDDING_CACHE_OVERRIDE is not None:
    EMBEDDING_CACHE_ROOT = Path(
        EMBEDDING_CACHE_OVERRIDE
    )

    assert _valid_erm_embedding_root(
        EMBEDDING_CACHE_ROOT
    ), (
        'Invalid EMBEDDING_CACHE_OVERRIDE: '
        f'{EMBEDDING_CACHE_ROOT}'
    )

else:
    embedding_candidates = []

    if EMBEDDING_CACHE_PARENT.exists():
        for train_file in EMBEDDING_CACHE_PARENT.rglob(
            'train_no_aug.pt'
        ):
            root = train_file.parent

            if _valid_erm_embedding_root(root):
                embedding_candidates.append(root)

    if not embedding_candidates:
        raise FileNotFoundError(
            'Could not find a valid Waterbirds-ERM embedding cache under:\n'
            f'{EMBEDDING_CACHE_PARENT}'
        )

    embedding_candidates = sorted(
        embedding_candidates,
        key=lambda root: (
            root / 'train_no_aug.pt'
        ).stat().st_mtime,
        reverse=True,
    )

    EMBEDDING_CACHE_ROOT = embedding_candidates[0]


train_cache_payload = torch.load(
    EMBEDDING_CACHE_ROOT / 'train_no_aug.pt',
    map_location='cpu',
    weights_only=False,
)

EMBED_METADATA = train_cache_payload.get(
    'cache_metadata',
    {},
)

EMBED_ERM_FINGERPRINT = (
    EMBED_METADATA.get(
        'erm_checkpoint_fingerprint'
    )
    or EMBED_METADATA.get(
        'erm_fingerprint'
    )
)


def _rater_candidate_info(path):
    try:
        checkpoint = torch.load(
            path,
            map_location='cpu',
            weights_only=False,
        )

        if (
            not isinstance(checkpoint, dict)
            or 'rater_sd' not in checkpoint
        ):
            return None

        if int(
            checkpoint.get(
                'feature_dim',
                2048,
            )
        ) != 2048:
            return None

        fingerprint = (
            checkpoint.get(
                'erm_fingerprint'
            )
            or checkpoint.get(
                'erm_checkpoint_fingerprint'
            )
        )

        return {
            'path': Path(path),
            'fingerprint': fingerprint,
            'mtime': Path(path).stat().st_mtime,
        }

    except Exception:
        return None


if RATER_CHECKPOINT_OVERRIDE is not None:
    RATER_CHECKPOINT = Path(
        RATER_CHECKPOINT_OVERRIDE
    )

    assert _rater_candidate_info(
        RATER_CHECKPOINT
    ) is not None

else:
    candidates = []

    for path in DRIVE_ROOT.rglob(
        'final_rater.pt'
    ):
        info = _rater_candidate_info(path)

        if info is not None:
            candidates.append(info)

    if not candidates:
        raise FileNotFoundError(
            'Could not find final_rater.pt under:\n'
            f'{DRIVE_ROOT}'
        )

    fingerprint_matches = [
        item
        for item in candidates
        if (
            EMBED_ERM_FINGERPRINT
            and item['fingerprint']
            == EMBED_ERM_FINGERPRINT
        )
    ]

    pool = (
        fingerprint_matches
        if fingerprint_matches
        else candidates
    )

    pool = sorted(
        pool,
        key=lambda item: item['mtime'],
        reverse=True,
    )

    RATER_CHECKPOINT = pool[0]['path']


RUNS_ROOT = (
    DRIVE_ROOT
    / 'frozen_new_rater_final_runs'
    / 'waterbirds_erm_embeddings'
)

RUNS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# DATA / FINAL CLASSIFIER
#
# Preserved from the ACTIVE attached notebook.
# ============================================================

SEED = 0
SPLIT_VAL = 0.6

BATCH_SIZE = 128
NUM_WORKERS = 2
FINAL_EPOCHS = 50

FINAL_LR = 1e-2
FINAL_MOMENTUM = 0.9
FINAL_WEIGHT_DECAY = 1e-4


# ============================================================
# WEIGHTING HYPERPARAMETERS
# ============================================================

NORMALIZATION_MODE = 'global'
WEIGHT_TRANSFORM = 'sigmoid'
WEIGHT_SCALE = 1.0
CORRECT_WEIGHT = 1.0
TEMPERATURE = 1.0

AUX_CE_COEFF = 0.0

SELECTION_METRIC = 'wga'
PLOT_EVERY = 1


def _tok(x):
    return str(x).replace('.', 'p')


RUN_NAME = (
    f'new_frozen_rater_ermemb_'
    f'{NORMALIZATION_MODE}_{WEIGHT_TRANSFORM}'
    f'_x{_tok(WEIGHT_SCALE)}'
    f'_auxce_{_tok(AUX_CE_COEFF)}'
    f'_splitval_{_tok(SPLIT_VAL)}'
    f'_lr_{_tok(FINAL_LR)}'
    f'_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
)

RUN_ROOT = (
    RUNS_ROOT
    / RUN_NAME
)

RUN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

LOG_PATH = RUN_ROOT / 'log.txt'
HISTORY_PATH = (
    RUN_ROOT
    / 'final_classifier_history.csv'
)

assert RATER_CHECKPOINT.exists()
assert EMBEDDING_CACHE_ROOT.exists()

print('Frozen Rater:')
print(RATER_CHECKPOINT)

print()
print('Waterbirds-ERM embedding cache:')
print(EMBEDDING_CACHE_ROOT)

print()
print('Embedding ERM fingerprint:')
print(EMBED_ERM_FINGERPRINT)

print()
print('Run directory:')
print(RUN_ROOT)

Mounted at /content/drive
Frozen Rater:
/content/drive/MyDrive/evidential_alignment_colab/rater_meta_only_runs/ea_gated_full_epoch_on_all_20261005_081758/final_rater.pt

Waterbirds-ERM embedding cache:
/content/drive/MyDrive/evidential_alignment_colab/embeddings_waterbirds_erm/waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224

Embedding ERM fingerprint:
1055b1e8de29

Run directory:
/content/drive/MyDrive/evidential_alignment_colab/frozen_new_rater_final_runs/waterbirds_erm_embeddings/new_frozen_rater_ermemb_global_sigmoid_x1p0_auxce_0p0_splitval_0p6_lr_0p01_20261005_082656


In [ ]:
# ============================================================
# 1. Reproducibility and logging
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

DEVICE = (
    'cuda'
    if torch.cuda.is_available()
    else 'cpu'
)

print(
    'Device:',
    DEVICE,
)


def log(msg=''):
    msg = str(msg)

    print(
        msg,
        flush=True,
    )

    with open(
        LOG_PATH,
        'a',
        encoding='utf-8',
    ) as f:
        f.write(
            msg
            + '\n'
        )


config = {
    'seed': SEED,
    'split_val': SPLIT_VAL,
    'batch_size': BATCH_SIZE,
    'final_epochs': FINAL_EPOCHS,
    'final_lr': FINAL_LR,
    'final_momentum': FINAL_MOMENTUM,
    'final_weight_decay': FINAL_WEIGHT_DECAY,
    'normalization_mode': NORMALIZATION_MODE,
    'weight_transform': WEIGHT_TRANSFORM,
    'weight_scale': WEIGHT_SCALE,
    'correct_weight': CORRECT_WEIGHT,
    'temperature': TEMPERATURE,
    'aux_ce_coeff': AUX_CE_COEFF,
    'selection_metric': SELECTION_METRIC,
    'rater_checkpoint': str(
        RATER_CHECKPOINT
    ),
    'embedding_cache_root': str(
        EMBEDDING_CACHE_ROOT
    ),
    'embedding_representation': (
        'waterbirds_erm_resnet50'
    ),
    'embedding_erm_fingerprint': (
        EMBED_ERM_FINGERPRINT
    ),
    'run_root': str(
        RUN_ROOT
    ),
}

with open(
    RUN_ROOT / 'run_config.json',
    'w',
    encoding='utf-8',
) as f:
    json.dump(
        config,
        f,
        indent=2,
    )


log('=' * 100)
log(
    'FROZEN NEW RATER / WATERBIRDS-ERM EMBEDDINGS'
)
log('=' * 100)
log(
    json.dumps(
        config,
        indent=2,
    )
)
log('=' * 100)

Device: cuda
FROZEN NEW RATER / WATERBIRDS-ERM EMBEDDINGS
{
  "seed": 0,
  "split_val": 0.6,
  "batch_size": 128,
  "final_epochs": 50,
  "final_lr": 0.01,
  "final_momentum": 0.9,
  "final_weight_decay": 0.0001,
  "normalization_mode": "global",
  "weight_transform": "sigmoid",
  "weight_scale": 1.0,
  "correct_weight": 1.0,
  "temperature": 1.0,
  "aux_ce_coeff": 0.0,
  "selection_metric": "wga",
  "rater_checkpoint": "/content/drive/MyDrive/evidential_alignment_colab/rater_meta_only_runs/ea_gated_full_epoch_on_all_20261005_081758/final_rater.pt",
  "embedding_cache_root": "/content/drive/MyDrive/evidential_alignment_colab/embeddings_waterbirds_erm/waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224",
  "embedding_representation": "waterbirds_erm_resnet50",
  "embedding_erm_fingerprint": "1055b1e8de29",
  "run_root": "/content/drive/MyDrive/evidential_alignment_colab/frozen_new_rater_final_runs/waterbirds_erm_embeddings/new_frozen_rater_ermemb_global_sigmoid_x1p0_auxce_0p0

In [ ]:
# ============================================================
# 2. Load persistent Waterbirds-ERM embeddings
# ============================================================

class IndexedEmbeddingDataset(Dataset):
    """
    Dataset returning:
        index, embedding, label, group, attr

    The index lets us reuse precomputed frozen-Rater scores even when
    the DataLoader shuffles calibration examples.
    """

    def __init__(
        self,
        embeddings,
        labels,
        groups,
        attrs,
        split_name,
    ):
        self.embeddings = embeddings.float()
        self.labels = labels.long()
        self.groups = groups.long()
        self.attrs = attrs.long()
        self.split_name = split_name

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, idx):
        return (
            idx,
            self.embeddings[idx],
            self.labels[idx],
            self.groups[idx],
            self.attrs[idx],
        )


def cache_token(x):
    return str(x).replace(
        '.',
        'p',
    )


def cache_file_for_split(
    split_name,
    split_val=SPLIT_VAL,
    seed=SEED,
):
    if split_name in {
        'train_no_aug',
        'val',
        'test',
    }:
        return (
            EMBEDDING_CACHE_ROOT
            / f'{split_name}.pt'
        )

    if split_name in {
        'val_subset1',
        'val_subset2',
    }:
        return (
            EMBEDDING_CACHE_ROOT
            / (
                f'{split_name}'
                f'_splittrain_1p0'
                f'_splitval_{cache_token(split_val)}'
                f'_seed_{seed}.pt'
            )
        )

    raise ValueError(
        f'Unknown split: {split_name}'
    )


def ensure_val_subset_embedding_cache(
    split_val=SPLIT_VAL,
    seed=SEED,
):
    """
    Create the exact-size val_subset1 / val_subset2 embedding tensors
    directly from the saved full val.pt if this split_val cache is absent.

    This follows the repository rule:
        indexes = arange(N)
        np.random.shuffle(indexes)
        n1 = int(N * split_val)

    No ResNet forward pass is performed.
    """

    p1 = cache_file_for_split(
        'val_subset1',
        split_val=split_val,
        seed=seed,
    )

    p2 = cache_file_for_split(
        'val_subset2',
        split_val=split_val,
        seed=seed,
    )

    if (
        p1.exists()
        and p2.exists()
    ):
        log(
            '[Embeddings] existing val subset cache found '
            f'for split_val={split_val}.'
        )
        return

    val_path = cache_file_for_split(
        'val'
    )

    if not val_path.exists():
        raise FileNotFoundError(
            'Missing full ERM val embeddings:\n'
            f'{val_path}'
        )

    payload = torch.load(
        val_path,
        map_location='cpu',
        weights_only=False,
    )

    required = {
        'embeddings',
        'labels',
        'groups',
        'attrs',
    }

    missing = (
        required
        - set(
            payload.keys()
        )
    )

    if missing:
        raise KeyError(
            f'{val_path} is missing keys: {missing}'
        )

    n = int(
        payload['labels'].shape[0]
    )

    rng = np.random.RandomState(
        seed
    )

    indexes = np.arange(
        n
    )

    rng.shuffle(
        indexes
    )

    n1 = int(
        n
        * split_val
    )

    index_sets = {
        'val_subset1': torch.as_tensor(
            indexes[:n1],
            dtype=torch.long,
        ),
        'val_subset2': torch.as_tensor(
            indexes[n1:],
            dtype=torch.long,
        ),
    }

    base_metadata = dict(
        payload.get(
            'cache_metadata',
            {},
        )
    )

    for split_name, idx in index_sets.items():
        target = cache_file_for_split(
            split_name,
            split_val=split_val,
            seed=seed,
        )

        subset = {
            'embeddings': payload[
                'embeddings'
            ][idx].clone(),
            'labels': payload[
                'labels'
            ][idx].clone(),
            'groups': payload[
                'groups'
            ][idx].clone(),
            'attrs': payload[
                'attrs'
            ][idx].clone(),
        }

        metadata = dict(
            base_metadata
        )

        metadata.update(
            {
                'split': split_name,
                'split_train': 1.0,
                'split_val': float(
                    split_val
                ),
                'seed': int(
                    seed
                ),
                'derived_from_full_val_cache': True,
            }
        )

        subset[
            'cache_metadata'
        ] = metadata

        torch.save(
            subset,
            target,
        )

        log(
            f'[Embeddings] created {split_name}: '
            f'n={len(idx)} -> {target}'
        )


ensure_val_subset_embedding_cache()


def load_embedding_split(split_name):
    path = cache_file_for_split(
        split_name
    )

    if not path.exists():
        raise FileNotFoundError(
            f'Missing cached embedding file:\n{path}'
        )

    payload = torch.load(
        path,
        map_location='cpu',
        weights_only=False,
    )

    required = {
        'embeddings',
        'labels',
        'groups',
        'attrs',
    }

    missing = (
        required
        - set(
            payload.keys()
        )
    )

    if missing:
        raise KeyError(
            f'{path} is missing keys: {missing}'
        )

    z = payload[
        'embeddings'
    ]

    if (
        z.ndim != 2
        or z.shape[1] != 2048
    ):
        raise ValueError(
            f'Unexpected embedding shape for {split_name}: '
            f'{tuple(z.shape)}'
        )

    metadata = payload.get(
        'cache_metadata',
        {},
    )

    representation = metadata.get(
        'representation_source'
    )

    if (
        representation
        and representation
        != 'waterbirds_erm_checkpoint'
    ):
        raise RuntimeError(
            'This cache is not the Waterbirds-ERM representation:\n'
            f'{path}'
        )

    ds = IndexedEmbeddingDataset(
        payload['embeddings'],
        payload['labels'],
        payload['groups'],
        payload['attrs'],
        split_name=split_name,
    )

    log(
        f'[Embeddings] loaded {split_name}: '
        f'{len(ds)} samples from {path}'
    )

    return ds


# Same final-classifier split structure as the attached notebook.
calibration_split = 'val_subset1'
selection_split = 'val_subset2'
test_split = 'test'

calibration_dataset = load_embedding_split(
    calibration_split
)

selection_dataset = load_embedding_split(
    selection_split
)

test_dataset = load_embedding_split(
    test_split
)


def summarize_groups(
    dataset,
    name,
):
    y = dataset.labels.numpy()
    g = dataset.groups.numpy()

    log('')
    log(
        f'{name}: n={len(dataset)}'
    )

    for gid in sorted(
        np.unique(g)
    ):
        mask = g == gid
        yy = y[mask]

        log(
            f'  group {gid}: '
            f'n={mask.sum()}, '
            f'class_mean={yy.mean():.3f}, '
            f'fraction={mask.mean():.3f}'
        )


summarize_groups(
    calibration_dataset,
    'Calibration / training',
)

summarize_groups(
    selection_dataset,
    'Selection',
)

summarize_groups(
    test_dataset,
    'Test',
)

[Embeddings] existing val subset cache found for split_val=0.6.
[Embeddings] loaded val_subset1: 719 samples from /content/drive/MyDrive/evidential_alignment_colab/embeddings_waterbirds_erm/waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p6_seed_0.pt
[Embeddings] loaded val_subset2: 480 samples from /content/drive/MyDrive/evidential_alignment_colab/embeddings_waterbirds_erm/waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset2_splittrain_1p0_splitval_0p6_seed_0.pt
[Embeddings] loaded test: 5794 samples from /content/drive/MyDrive/evidential_alignment_colab/embeddings_waterbirds_erm/waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/test.pt

Calibration / training: n=719
  group 0: n=272, class_mean=0.000, fraction=0.378
  group 1: n=281, class_mean=0.000, fraction=0.391
  group 2: n=78, class_mean=1.000, fraction=0.108
  group 3: n=88, class_mean=1.000, fraction=0.122

Selection: n=480
  group 0: n=195

In [ ]:
# ============================================================
# 3. Define and load the NEW frozen trained Rater
#
# Rater optimization: NONE
# Meta-learning: NONE
# Inner models: NONE
# ============================================================

class EmbeddingRaterSmall(nn.Module):
    def __init__(
        self,
        input_dim=2048,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 1),
        )

    def forward(self, z):
        return self.net(z).squeeze(-1)


class EmbeddingRaterMedium(nn.Module):
    def __init__(
        self,
        input_dim=2048,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, 512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Linear(64, 1),
        )

    def forward(self, z):
        return self.net(z).squeeze(-1)


checkpoint = torch.load(
    RATER_CHECKPOINT,
    map_location='cpu',
    weights_only=False,
)

if not isinstance(
    checkpoint,
    dict,
):
    raise TypeError(
        'Expected final_rater.pt to be a checkpoint dictionary.'
    )

state_dict = checkpoint.get(
    'rater_sd'
)

if state_dict is None:
    raise KeyError(
        'final_rater.pt does not contain rater_sd.'
    )

capacity = str(
    checkpoint.get(
        'rater_capacity',
        '',
    )
).lower()

if capacity not in {
    'small',
    'medium',
}:
    first_hidden = state_dict.get(
        'net.1.weight'
    )

    if (
        first_hidden is not None
        and first_hidden.shape[0] == 256
    ):
        capacity = 'small'

    elif (
        first_hidden is not None
        and first_hidden.shape[0] == 512
    ):
        capacity = 'medium'

    else:
        raise ValueError(
            'Could not infer Rater capacity.'
        )


if capacity == 'small':
    rater = EmbeddingRaterSmall(
        2048
    )
else:
    rater = EmbeddingRaterMedium(
        2048
    )


rater.load_state_dict(
    state_dict
)

rater.to(
    DEVICE
)

rater.eval()

for p in rater.parameters():
    p.requires_grad_(False)

num_trainable = sum(
    p.numel()
    for p in rater.parameters()
    if p.requires_grad
)

assert num_trainable == 0


checkpoint_weighting = (
    checkpoint.get('weighting')
    or checkpoint.get('rater_weighting')
)

checkpoint_temperature = (
    checkpoint.get('temperature')
    or checkpoint.get('rater_temperature')
)

checkpoint_fp = (
    checkpoint.get('erm_fingerprint')
    or checkpoint.get(
        'erm_checkpoint_fingerprint'
    )
)


log('')
log('Frozen NEW Rater loaded:')
log(
    f'  checkpoint = {RATER_CHECKPOINT}'
)
log(
    f'  capacity = {capacity}'
)
log(
    f'  checkpoint weighting = {checkpoint_weighting}'
)
log(
    f'  checkpoint temperature = {checkpoint_temperature}'
)
log(
    f'  checkpoint ERM fingerprint = {checkpoint_fp}'
)
log(
    f'  embedding ERM fingerprint = {EMBED_ERM_FINGERPRINT}'
)
log(
    '  Rater optimization = NONE'
)
log(
    '  Meta-learning = NONE'
)
log(
    '  Inner models = NONE'
)
log(
    '  trainable Rater params = 0'
)


if (
    checkpoint_fp
    and EMBED_ERM_FINGERPRINT
    and checkpoint_fp != EMBED_ERM_FINGERPRINT
):
    raise RuntimeError(
        'Frozen Rater and embedding cache use different ERM fingerprints.'
    )


if (
    checkpoint_weighting
    and str(
        checkpoint_weighting
    ).lower()
    != WEIGHT_TRANSFORM.lower()
):
    log(
        'WARNING: WEIGHT_TRANSFORM differs from '
        'the Rater checkpoint metadata.'
    )


if (
    checkpoint_temperature is not None
    and abs(
        float(
            checkpoint_temperature
        )
        - float(
            TEMPERATURE
        )
    ) > 1e-12
):
    log(
        'WARNING: TEMPERATURE differs from '
        'the Rater checkpoint metadata.'
    )


Frozen NEW Rater loaded:
  checkpoint = /content/drive/MyDrive/evidential_alignment_colab/rater_meta_only_runs/ea_gated_full_epoch_on_all_20261005_081758/final_rater.pt
  capacity = medium
  checkpoint weighting = sigmoid
  checkpoint temperature = 1.0
  checkpoint ERM fingerprint = 1055b1e8de29
  embedding ERM fingerprint = 1055b1e8de29
  Rater optimization = NONE
  Meta-learning = NONE
  Inner models = NONE
  trainable Rater params = 0


In [ ]:
# ============================================================
# 4. Precompute NEW frozen-Rater RAW scores once
# ============================================================
#
# We compute the Rater score for every calibration example ONCE.
#
# Raw Rater scores are constant because both embeddings and Rater are frozen.
# The CURRENT wrong/correct mask and wrong-batch normalization remain dynamic.
# ============================================================

@torch.no_grad()
def compute_raw_rater_scores(dataset, batch_size=BATCH_SIZE):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    raw_scores = torch.empty(len(dataset), dtype=torch.float32)

    for idx, z, y, g, a in loader:
        z = z.to(DEVICE, non_blocking=True)

        s = rater(z).detach().cpu()

        raw_scores[idx.long()] = s

    return raw_scores


calibration_raw_scores = compute_raw_rater_scores(
    calibration_dataset
)

mu_R = calibration_raw_scores.mean().item()
sigma_R = calibration_raw_scores.std(unbiased=False).item()

if sigma_R < 1e-8:
    raise RuntimeError(
        "Calibration Rater-score std is too close to zero."
    )

log("")
log("[Global Rater statistics on calibration set]")
log(f"mu_R    = {mu_R:.6f}")
log(f"sigma_R = {sigma_R:.6f}")
log(f"min     = {calibration_raw_scores.min().item():.6f}")
log(f"max     = {calibration_raw_scores.max().item():.6f}")


# ------------------------------------------------------------
# Save score table and histogram
# ------------------------------------------------------------

score_dir = RUN_ROOT / "rater_score_diagnostics"
score_dir.mkdir(parents=True, exist_ok=True)

with open(score_dir / "calibration_raw_rater_scores.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["index", "raw_score", "label", "group", "attr"])

    for i in range(len(calibration_dataset)):
        writer.writerow([
            i,
            float(calibration_raw_scores[i].item()),
            int(calibration_dataset.labels[i].item()),
            int(calibration_dataset.groups[i].item()),
            int(calibration_dataset.attrs[i].item()),
        ])


def group_name(gid):
    names = {
        0: "Group 0: landbird / land",
        1: "Group 1: landbird / water",
        2: "Group 2: waterbird / land",
        3: "Group 3: waterbird / water",
    }
    return names.get(int(gid), f"Group {int(gid)}")


fig, ax = plt.subplots(figsize=(10, 7))
groups_np = calibration_dataset.groups.numpy()
scores_np = calibration_raw_scores.numpy()

for gid in sorted(np.unique(groups_np)):
    mask = groups_np == gid
    ax.hist(
        scores_np[mask],
        bins=40,
        density=True,
        alpha=0.45,
        label=f"{group_name(gid)} (n={mask.sum()})",
    )

ax.axvline(mu_R, linestyle="--", linewidth=1, label="global mean")
ax.set_title("New frozen-Rater raw score distribution on calibration set")
ax.set_xlabel("Raw Rater score")
ax.set_ylabel("Density")
ax.legend(fontsize=9)
ax.grid(True, linestyle=":", alpha=0.3)
fig.tight_layout()
fig.savefig(score_dir / "calibration_raw_score_histogram_by_group.png", dpi=160)
plt.close(fig)

log(f"Saved calibration Rater-score diagnostics to: {score_dir}")



[Global Rater statistics on calibration set]
mu_R    = -0.171048
sigma_R = 0.181635
min     = -0.624651
max     = 0.301214
Saved calibration Rater-score diagnostics to: /content/drive/MyDrive/evidential_alignment_colab/frozen_new_rater_final_runs/waterbirds_erm_embeddings/new_frozen_rater_ermemb_global_sigmoid_x1p0_auxce_0p0_splitval_0p6_lr_0p01_20261005_082656/rater_score_diagnostics


In [ ]:
# ============================================================
# 5. Final classifier and evaluation helpers
# ============================================================

class FinalLinearClassifier(nn.Module):
    def __init__(self, input_dim=2048, num_classes=2):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, z):
        return self.linear(z)


@torch.no_grad()
def evaluate_classifier(model, dataset, batch_size=BATCH_SIZE):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_n = 0

    group_correct = {}
    group_total = {}

    for idx, z, y, g, a in loader:
        z = z.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        g = g.to(DEVICE, non_blocking=True)

        logits = model(z)
        loss = F.cross_entropy(logits, y, reduction="sum")

        pred = logits.argmax(dim=1)

        total_loss += loss.item()
        total_correct += (pred == y).sum().item()
        total_n += y.numel()

        for gid_tensor in torch.unique(g):
            gid = int(gid_tensor.item())
            mask = g == gid_tensor

            group_correct[gid] = (
                group_correct.get(gid, 0)
                + (pred[mask] == y[mask]).sum().item()
            )
            group_total[gid] = (
                group_total.get(gid, 0)
                + mask.sum().item()
            )

    group_accuracy = {
        gid: group_correct[gid] / group_total[gid]
        for gid in sorted(group_total)
    }

    return {
        "loss": total_loss / max(total_n, 1),
        "accuracy": total_correct / max(total_n, 1),
        "group_accuracy": group_accuracy,
        "worst_group_accuracy": min(group_accuracy.values()),
    }


def selection_value(metrics):
    if SELECTION_METRIC == "wga":
        return metrics["worst_group_accuracy"]
    if SELECTION_METRIC == "accuracy":
        return metrics["accuracy"]
    if SELECTION_METRIC == "loss":
        return -metrics["loss"]
    raise ValueError(
        "SELECTION_METRIC must be wga, accuracy, or loss."
    )


def format_group_acc(metrics):
    return ", ".join(
        f"g{gid}={100.0 * acc:.2f}%"
        for gid, acc in metrics["group_accuracy"].items()
    )


In [ ]:
# ============================================================
# 6. Dynamic weighting rule
#
# BASELINE:
#   NORMALIZATION_MODE = 'wrong_batch'
#   WEIGHT_TRANSFORM   = 'sigmoid'
#   WEIGHT_SCALE       = 1.0
#   CORRECT_WEIGHT     = 1.0
# ============================================================


def transform_wrong_scores(raw_scores):
    # raw_scores contains ONLY the CURRENTLY MISCLASSIFIED
    # examples in the CURRENT minibatch.
    if raw_scores.numel() == 0:
        return raw_scores

    # --------------------------------------------------------
    # Normalization
    # --------------------------------------------------------
    if NORMALIZATION_MODE == 'wrong_batch':
        # EXACT old frozen-rater.py baseline behavior.
        score_mean = raw_scores.mean()
        score_std = raw_scores.std(unbiased=False)
        normalized = (
            raw_scores - score_mean
        ) / (
            score_std + 1e-6
        )

    elif NORMALIZATION_MODE == 'global':
        # Experimental fixed calibration-wide normalization.
        normalized = (
            raw_scores - mu_R
        ) / (
            sigma_R + 1e-6
        )

    elif NORMALIZATION_MODE == 'none':
        normalized = raw_scores

    else:
        raise ValueError(
            f'Unknown NORMALIZATION_MODE: {NORMALIZATION_MODE}'
        )

    # --------------------------------------------------------
    # Transform
    # --------------------------------------------------------
    if WEIGHT_TRANSFORM == 'sigmoid':
        ratings = torch.sigmoid(
            normalized / TEMPERATURE
        )

    elif WEIGHT_TRANSFORM == 'softmax':
        # Softmax among CURRENTLY WRONG examples only.
        centered = normalized - normalized.mean()
        ratings = torch.softmax(
            centered / TEMPERATURE,
            dim=0,
        )

    else:
        raise ValueError(
            f'Unknown WEIGHT_TRANSFORM: {WEIGHT_TRANSFORM}'
        )

    return WEIGHT_SCALE * ratings


@torch.no_grad()
def compute_batch_weights(
    batch_indices,
    logits,
    y,
):
    # DYNAMIC every minibatch: predictions come from the CURRENT
    # final classifier, so the wrong/correct set changes through training.
    pred = logits.argmax(dim=1)
    correct_mask = pred.eq(y)
    wrong_mask = ~correct_mask

    weights = torch.full(
        size=(y.shape[0],),
        fill_value=float(CORRECT_WEIGHT),
        device=logits.device,
        dtype=logits.dtype,
    )

    raw_wrong_scores = torch.empty(
        0,
        device=logits.device,
        dtype=logits.dtype,
    )

    wrong_weights = torch.empty(
        0,
        device=logits.device,
        dtype=logits.dtype,
    )

    if wrong_mask.any():
        idx_cpu = batch_indices.detach().cpu().long()
        wrong_cpu = wrong_mask.detach().cpu()

        # Raw scores were cached once because the Rater is frozen.
        # Only the CURRENT wrong subset is selected here.
        raw_wrong_scores = calibration_raw_scores[
            idx_cpu[wrong_cpu]
        ].to(logits.device)

        wrong_weights = transform_wrong_scores(
            raw_wrong_scores
        )

        weights[wrong_mask] = wrong_weights

    return (
        weights,
        correct_mask,
        wrong_mask,
        raw_wrong_scores,
        wrong_weights,
    )


is_exact_baseline = (
    NORMALIZATION_MODE == 'wrong_batch'
    and WEIGHT_TRANSFORM == 'sigmoid'
    and WEIGHT_SCALE == 1.0
    and CORRECT_WEIGHT == 1.0
    and AUX_CE_COEFF == 0.0
    and FINAL_LR == 1e-2
    and FINAL_MOMENTUM == 0.9
    and FINAL_WEIGHT_DECAY == 1e-4
)

log('')
if is_exact_baseline:
    log('BASELINE MODE DETECTED')
    log('  current wrong-batch mean/std')
    log('  wrong -> sigmoid')
    log('  correct -> 1')
    log('  weighted CE sum only')
else:
    log('NON-BASELINE WEIGHTING EXPERIMENT DETECTED')



NON-BASELINE WEIGHTING EXPERIMENT DETECTED


In [ ]:
# ============================================================
# 7. Weight diagnostics
# ============================================================

plot_root = RUN_ROOT / 'plots'
weight_hist_dir = plot_root / 'epoch_actual_weight_histogram'
wrong_hist_dir = plot_root / 'epoch_wrong_only_weight_histogram'
data_dir = plot_root / 'epoch_weight_data'

for d in [weight_hist_dir, wrong_hist_dir, data_dir]:
    d.mkdir(parents=True, exist_ok=True)


def save_epoch_weight_plots(
    epoch,
    weights,
    groups,
    correctness,
    wrong_weights,
    wrong_groups,
):
    weights = np.asarray(weights, dtype=np.float64)
    groups = np.asarray(groups, dtype=np.int64)
    correctness = np.asarray(correctness, dtype=np.float64)
    wrong_weights = np.asarray(wrong_weights, dtype=np.float64)
    wrong_groups = np.asarray(wrong_groups, dtype=np.int64)

    csv_path = data_dir / f'epoch_{epoch:03d}_calibration_weights.csv'
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            'index_in_epoch_order',
            'actual_weight',
            'group',
            'correct_before_update',
        ])
        for i in range(len(weights)):
            writer.writerow([
                i,
                float(weights[i]),
                int(groups[i]),
                int(correctness[i] >= 0.5),
            ])

    max_x = max(1.05, float(weights.max()) + 0.05)
    bins = np.linspace(0.0, max_x, 50)

    fig, ax = plt.subplots(figsize=(10, 7))
    for gid in sorted(np.unique(groups)):
        mask = groups == gid
        ax.hist(
            weights[mask],
            bins=bins,
            density=True,
            alpha=0.45,
            label=f'g{gid} (n={mask.sum()})',
        )

    ax.axvline(1.0, linestyle='--', linewidth=1, label='weight = 1')
    ax.set_title(f'Actual calibration weights | epoch {epoch}')
    ax.set_xlabel('Actual training weight')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    ax.grid(True, linestyle=':', alpha=0.3)
    fig.tight_layout()
    fig.savefig(
        weight_hist_dir / f'epoch_{epoch:03d}_actual_weight_hist.png',
        dpi=160,
        bbox_inches='tight',
    )
    plt.close(fig)

    if len(wrong_weights) > 0:
        max_wrong_x = max(1.05, float(wrong_weights.max()) + 0.05)
        wrong_bins = np.linspace(0.0, max_wrong_x, 50)

        fig, ax = plt.subplots(figsize=(10, 7))
        for gid in sorted(np.unique(wrong_groups)):
            mask = wrong_groups == gid
            ax.hist(
                wrong_weights[mask],
                bins=wrong_bins,
                density=True,
                alpha=0.45,
                label=f'g{gid} (wrong n={mask.sum()})',
            )

        ax.axvline(1.0, linestyle='--', linewidth=1, label='weight = 1')
        ax.set_title(f'Rater weights for CURRENT mistakes | epoch {epoch}')
        ax.set_xlabel('Wrong-sample Rater-derived weight')
        ax.set_ylabel('Density')
        ax.legend(fontsize=9)
        ax.grid(True, linestyle=':', alpha=0.3)
        fig.tight_layout()
        fig.savefig(
            wrong_hist_dir / f'epoch_{epoch:03d}_wrong_only_weight_hist.png',
            dpi=160,
            bbox_inches='tight',
        )
        plt.close(fig)


In [ ]:
# ============================================================
# 8. Train fresh final classifier
#
# This version tracks TWO different things:
#
# 1. VALID selection:
#       best model according to selection/validation metric
#
# 2. TEST ORACLE DIAGNOSTIC:
#       highest test WGA observed at ANY epoch
#
# The test-oracle result is for analysis only.
# It is NOT used to restore/select the final model.
# ============================================================

# Use the same seed for each weighting experiment so comparisons
# start from the same fresh Linear(2048 -> 2) initialization.
set_seed(SEED)

model = FinalLinearClassifier(
    input_dim=2048,
    num_classes=2,
).to(DEVICE)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=FINAL_LR,
    momentum=FINAL_MOMENTUM,
    weight_decay=FINAL_WEIGHT_DECAY,
)


# ============================================================
# Reproducible calibration DataLoader
# ============================================================

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

cal_loader = DataLoader(
    calibration_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    generator=loader_generator,
)


# ============================================================
# A. VALIDATION / SELECTION-BEST tracking
# ============================================================

best_value = -float('inf')

best_epoch = None
best_state = None

best_selection_metrics = None
best_test_metrics = None


# ============================================================
# B. TEST-ORACLE tracking
#
# IMPORTANT:
# This is DIAGNOSTIC ONLY.
#
# We are intentionally asking:
#
#   "What is the largest test WGA this training trajectory
#    ever reached?"
#
# It must NOT replace validation-based model selection.
# ============================================================

best_test_wga_all = -float('inf')

best_test_epoch_all = None
best_test_state_all = None

best_test_metrics_all = None
best_test_selection_metrics_all = None
best_test_calibration_metrics_all = None


# ============================================================
# History
# ============================================================

history_rows = []


# ============================================================
# Logging
# ============================================================

log('')
log('=' * 100)
log('TRAINING FRESH FINAL CLASSIFIER WITH NEW FROZEN RATER')
log('=' * 100)

log('Architecture: Linear(2048 -> 2)')

log(
    f'Calibration split: '
    f'{calibration_split}, '
    f'n={len(calibration_dataset)}'
)

log(
    f'Selection split:   '
    f'{selection_split}, '
    f'n={len(selection_dataset)}'
)

log(
    f'Test split:        '
    f'{test_split}, '
    f'n={len(test_dataset)}'
)

log('')

log(
    f'Normalization: '
    f'{NORMALIZATION_MODE}'
)

log(
    f'Transform: '
    f'{WEIGHT_TRANSFORM}'
)

log(
    f'Weight scale: '
    f'{WEIGHT_SCALE}'
)

log(
    f'Correct weight: '
    f'{CORRECT_WEIGHT}'
)

log(
    f'Temperature: '
    f'{TEMPERATURE}'
)

log(
    f'Aux ordinary CE coefficient: '
    f'{AUX_CE_COEFF}'
)

log('')

log(
    f'LR={FINAL_LR}, '
    f'momentum={FINAL_MOMENTUM}, '
    f'weight_decay={FINAL_WEIGHT_DECAY}'
)

log(
    'Weighted CE uses SUM and is NOT divided by sum(weights).'
)

log('')

log(
    'NOTE: highest test WGA across epochs will also be tracked '
    'as an ORACLE DIAGNOSTIC only.'
)

log(
    'The actual final model is still selected ONLY by the '
    'selection/validation metric.'
)

log('=' * 100)


# ============================================================
# Training loop
# ============================================================

for epoch in range(
    1,
    FINAL_EPOCHS + 1,
):

    model.train()

    running_weighted_ce = 0.0
    running_plain_ce = 0.0
    running_total_loss = 0.0

    num_batches = 0

    online_correct = 0
    online_total = 0

    epoch_weights = []
    epoch_groups = []
    epoch_correctness = []

    epoch_wrong_weights = []
    epoch_wrong_groups = []


    # ========================================================
    # Calibration training
    # ========================================================

    for (
        batch_indices,
        z,
        y,
        g,
        a,
    ) in cal_loader:

        z = z.to(
            DEVICE,
            non_blocking=True,
        )

        y = y.to(
            DEVICE,
            non_blocking=True,
        )


        # ----------------------------------------------------
        # CURRENT classifier prediction
        # ----------------------------------------------------

        logits = model(z)


        # ----------------------------------------------------
        # CURRENT correct/wrong mask
        # and CURRENT Rater-derived weights
        # ----------------------------------------------------

        with torch.no_grad():

            (
                weights,
                correct_mask,
                wrong_mask,
                raw_wrong_scores,
                wrong_weights,
            ) = compute_batch_weights(
                batch_indices,
                logits,
                y,
            )


        # ----------------------------------------------------
        # Per-sample CE
        # ----------------------------------------------------

        per_sample_ce = F.cross_entropy(
            logits,
            y,
            reduction='none',
        )


        # ----------------------------------------------------
        # Weighted CE
        #
        # SAME definition as before:
        #
        #     SUM_i w_i * CE_i
        #
        # NO division by sum(weights)
        # ----------------------------------------------------

        weighted_ce_loss = (
            per_sample_ce
            * weights
        ).sum()


        # ----------------------------------------------------
        # Optional ordinary CE
        # ----------------------------------------------------

        plain_ce_loss = (
            per_sample_ce.sum()
        )


        # ----------------------------------------------------
        # Total loss
        # ----------------------------------------------------

        total_loss = (
            weighted_ce_loss
            + AUX_CE_COEFF
            * plain_ce_loss
        )


        optimizer.zero_grad()

        total_loss.backward()

        optimizer.step()


        # ----------------------------------------------------
        # Running statistics
        # ----------------------------------------------------

        running_weighted_ce += (
            weighted_ce_loss.item()
        )

        running_plain_ce += (
            plain_ce_loss.item()
        )

        running_total_loss += (
            total_loss.item()
        )

        num_batches += 1


        online_correct += (
            correct_mask.sum().item()
        )

        online_total += (
            y.numel()
        )


        # ----------------------------------------------------
        # Save weights for plots
        # ----------------------------------------------------

        epoch_weights.append(
            weights.detach().cpu()
        )

        epoch_groups.append(
            g.detach().cpu()
        )

        epoch_correctness.append(
            correct_mask
            .float()
            .detach()
            .cpu()
        )


        wrong_cpu = (
            wrong_mask
            .detach()
            .cpu()
        )

        if wrong_cpu.any():

            epoch_wrong_weights.append(
                wrong_weights
                .detach()
                .cpu()
            )

            epoch_wrong_groups.append(
                g[
                    wrong_cpu
                ]
                .detach()
                .cpu()
            )


    # ========================================================
    # Combine epoch weight statistics
    # ========================================================

    weights_np = (
        torch.cat(
            epoch_weights
        )
        .numpy()
    )

    groups_np = (
        torch.cat(
            epoch_groups
        )
        .numpy()
    )

    correctness_np = (
        torch.cat(
            epoch_correctness
        )
        .numpy()
    )


    if epoch_wrong_weights:

        wrong_weights_np = (
            torch.cat(
                epoch_wrong_weights
            )
            .numpy()
        )

        wrong_groups_np = (
            torch.cat(
                epoch_wrong_groups
            )
            .numpy()
        )

    else:

        wrong_weights_np = np.array(
            [],
            dtype=np.float64,
        )

        wrong_groups_np = np.array(
            [],
            dtype=np.int64,
        )


    # ========================================================
    # Stable post-epoch evaluation
    # ========================================================

    calibration_metrics = evaluate_classifier(
        model,
        calibration_dataset,
    )

    selection_metrics = evaluate_classifier(
        model,
        selection_dataset,
    )

    test_metrics = evaluate_classifier(
        model,
        test_dataset,
    )


    # Selection criterion used for the REAL final model.
    value = selection_value(
        selection_metrics
    )


    # ========================================================
    # NEW:
    # Track highest TEST WGA ever observed
    #
    # This happens independently of validation behavior.
    # ========================================================

    current_test_wga = float(
        test_metrics[
            'worst_group_accuracy'
        ]
    )


    if (
        current_test_wga
        > best_test_wga_all
    ):

        best_test_wga_all = (
            current_test_wga
        )

        best_test_epoch_all = (
            epoch
        )

        best_test_state_all = {
            k: (
                v
                .detach()
                .cpu()
                .clone()
            )
            for k, v
            in model
            .state_dict()
            .items()
        }

        best_test_metrics_all = (
            test_metrics
        )

        best_test_selection_metrics_all = (
            selection_metrics
        )

        best_test_calibration_metrics_all = (
            calibration_metrics
        )


        # ----------------------------------------------------
        # Save oracle-test checkpoint separately.
        #
        # DO NOT confuse this with the selection-best model.
        # ----------------------------------------------------

        torch.save(
            {
                'model_sd': (
                    best_test_state_all
                ),

                'best_test_epoch': (
                    best_test_epoch_all
                ),

                'best_test_wga': (
                    best_test_wga_all
                ),

                'test_metrics': (
                    best_test_metrics_all
                ),

                'selection_metrics_at_test_best': (
                    best_test_selection_metrics_all
                ),

                'calibration_metrics_at_test_best': (
                    best_test_calibration_metrics_all
                ),

                'rater_checkpoint': str(
                    RATER_CHECKPOINT
                ),

                'config': config,

                'warning': (
                    'TEST-ORACLE DIAGNOSTIC ONLY. '
                    'Do not use this checkpoint as '
                    'validation-selected final performance.'
                ),
            },

            RUN_ROOT
            / 'best_test_wga_oracle_diagnostic.pt',
        )


        log(
            f'[BEST TEST WGA OBSERVED] '
            f'epoch={best_test_epoch_all:03d}, '
            f'test_WGA='
            f'{100.0 * best_test_wga_all:.2f}%, '
            f'selection_WGA='
            f'{100.0 * selection_metrics["worst_group_accuracy"]:.2f}%'
        )


    # ========================================================
    # Save epoch weight plots
    # ========================================================

    if (
        epoch == 1
        or epoch % PLOT_EVERY == 0
        or epoch == FINAL_EPOCHS
    ):

        save_epoch_weight_plots(
            epoch,
            weights_np,
            groups_np,
            correctness_np,
            wrong_weights_np,
            wrong_groups_np,
        )


    # ========================================================
    # Other epoch statistics
    # ========================================================

    online_acc = (
        online_correct
        / max(
            online_total,
            1,
        )
    )


    mean_wrong = (
        float(
            wrong_weights_np.mean()
        )
        if len(
            wrong_weights_np
        )
        else float('nan')
    )


    # ========================================================
    # History
    #
    # Two new columns:
    #
    #   best_test_wga_so_far
    #   best_test_wga_epoch_so_far
    # ========================================================

    history_rows.append(
        [
            epoch,

            running_weighted_ce
            / max(
                num_batches,
                1,
            ),

            running_plain_ce
            / max(
                num_batches,
                1,
            ),

            running_total_loss
            / max(
                num_batches,
                1,
            ),

            online_acc,

            calibration_metrics[
                'loss'
            ],

            calibration_metrics[
                'accuracy'
            ],

            calibration_metrics[
                'worst_group_accuracy'
            ],

            selection_metrics[
                'loss'
            ],

            selection_metrics[
                'accuracy'
            ],

            selection_metrics[
                'worst_group_accuracy'
            ],

            test_metrics[
                'loss'
            ],

            test_metrics[
                'accuracy'
            ],

            test_metrics[
                'worst_group_accuracy'
            ],

            # NEW
            best_test_wga_all,

            # NEW
            best_test_epoch_all,

            float(
                weights_np.mean()
            ),

            mean_wrong,

            int(
                len(
                    wrong_weights_np
                )
            ),
        ]
    )


    # ========================================================
    # Rewrite history CSV
    # ========================================================

    with open(
        HISTORY_PATH,
        'w',
        newline='',
    ) as f:

        writer = csv.writer(
            f
        )

        writer.writerow(
            [
                'epoch',

                'weighted_ce_batch_avg',

                'plain_ce_batch_avg',

                'total_loss_batch_avg',

                'online_calibration_accuracy',

                'calibration_loss',

                'calibration_accuracy',

                'calibration_wga',

                'selection_loss',

                'selection_accuracy',

                'selection_wga',

                'test_loss',

                'test_accuracy',

                'test_wga',

                # NEW
                'best_test_wga_so_far',

                # NEW
                'best_test_wga_epoch_so_far',

                'mean_actual_weight',

                'mean_wrong_weight',

                'wrong_seen_this_epoch',
            ]
        )

        writer.writerows(
            history_rows
        )


    # ========================================================
    # Epoch log
    # ========================================================

    log(
        f'[Epoch {epoch:03d}] '

        f'online_cal_acc='
        f'{100.0 * online_acc:.2f}%, '

        f'cal_eval_acc='
        f'{100.0 * calibration_metrics["accuracy"]:.2f}%, '

        f'cal_WGA='
        f'{100.0 * calibration_metrics["worst_group_accuracy"]:.2f}%, '

        f'selection_acc='
        f'{100.0 * selection_metrics["accuracy"]:.2f}%, '

        f'selection_WGA='
        f'{100.0 * selection_metrics["worst_group_accuracy"]:.2f}%, '

        f'test_acc='
        f'{100.0 * test_metrics["accuracy"]:.2f}%, '

        f'test_WGA='
        f'{100.0 * test_metrics["worst_group_accuracy"]:.2f}%, '

        f'best_test_WGA_so_far='
        f'{100.0 * best_test_wga_all:.2f}%'
        f'@{best_test_epoch_all:03d}, '

        f'mean_w='
        f'{weights_np.mean():.4f}, '

        f'mean_wrong_w='
        f'{mean_wrong:.4f}, '

        f'wrong_seen='
        f'{len(wrong_weights_np)}'
    )


    log(
        f'[Epoch {epoch:03d} selection groups] '
        f'{format_group_acc(selection_metrics)}'
    )


    log(
        f'[Epoch {epoch:03d} test groups] '
        f'{format_group_acc(test_metrics)}'
    )


    # ========================================================
    # VALID model selection
    #
    # This remains unchanged:
    # selection/validation determines the real final model.
    # ========================================================

    if value > best_value:

        best_value = float(
            value
        )

        best_epoch = (
            epoch
        )

        best_state = {
            k: (
                v
                .detach()
                .cpu()
                .clone()
            )
            for k, v
            in model
            .state_dict()
            .items()
        }

        best_selection_metrics = (
            selection_metrics
        )

        best_test_metrics = (
            test_metrics
        )


        torch.save(
            {
                'model_sd': (
                    best_state
                ),

                'best_epoch': (
                    best_epoch
                ),

                'selection_metric': (
                    SELECTION_METRIC
                ),

                'selection_value': (
                    best_value
                ),

                'selection_metrics': (
                    best_selection_metrics
                ),

                'test_metrics_at_selection_time': (
                    best_test_metrics
                ),

                'rater_checkpoint': str(
                    RATER_CHECKPOINT
                ),

                'config': config,
            },

            RUN_ROOT
            / 'best_final_classifier.pt',
        )


        log(
            f'[BEST UPDATED] '
            f'epoch={best_epoch:03d}, '

            f'selection_WGA='
            f'{100.0 * selection_metrics["worst_group_accuracy"]:.2f}%, '

            f'test_WGA='
            f'{100.0 * test_metrics["worst_group_accuracy"]:.2f}%'
        )


# ============================================================
# Restore VALIDATION-SELECTION-BEST classifier
#
# IMPORTANT:
# We do NOT restore the test-oracle checkpoint.
# ============================================================

assert best_state is not None

model.load_state_dict(
    best_state
)

model.to(
    DEVICE
)

model.eval()


# ============================================================
# Save restored valid final classifier
# ============================================================

torch.save(
    {
        'model_sd': (
            model.state_dict()
        ),

        'best_epoch': (
            best_epoch
        ),

        'selection_metric': (
            SELECTION_METRIC
        ),

        'selection_value': (
            best_value
        ),

        'selection_metrics': (
            best_selection_metrics
        ),

        'test_metrics_at_selection_time': (
            best_test_metrics
        ),

        # Also record oracle information,
        # but do NOT use it for selection.
        'best_test_wga_observed': (
            best_test_wga_all
        ),

        'best_test_wga_epoch': (
            best_test_epoch_all
        ),

        'rater_checkpoint': str(
            RATER_CHECKPOINT
        ),

        'config': config,
    },

    RUN_ROOT
    / 'final_classifier_restored_best.pt',
)


# ============================================================
# Final report
# ============================================================

log('')
log('=' * 100)
log('TRAINING COMPLETE')
log('=' * 100)


# ------------------------------------------------------------
# VALID result
# ------------------------------------------------------------

log(
    'VALIDATION-SELECTED RESULT'
)

log(
    f'Best epoch selected by '
    f'{SELECTION_METRIC}: '
    f'{best_epoch}'
)

log(
    f'Best selection WGA: '
    f'{100.0 * best_selection_metrics["worst_group_accuracy"]:.2f}%'
)

log(
    f'Test WGA at selected epoch: '
    f'{100.0 * best_test_metrics["worst_group_accuracy"]:.2f}%'
)

log(
    f'Test groups at selected epoch: '
    f'{format_group_acc(best_test_metrics)}'
)


# ------------------------------------------------------------
# TEST ORACLE / DIAGNOSTIC result
# ------------------------------------------------------------

log('')

log(
    'TEST-ORACLE DIAGNOSTIC '
    '(NOT FOR MODEL SELECTION)'
)

log(
    f'Highest test WGA observed over ALL epochs: '
    f'{100.0 * best_test_wga_all:.2f}%'
)

log(
    f'Epoch of highest observed test WGA: '
    f'{best_test_epoch_all}'
)

log(
    f'Selection WGA at that epoch: '
    f'{100.0 * best_test_selection_metrics_all["worst_group_accuracy"]:.2f}%'
)

log(
    f'Test accuracy at that epoch: '
    f'{100.0 * best_test_metrics_all["accuracy"]:.2f}%'
)

log(
    f'Test groups at test-oracle epoch: '
    f'{format_group_acc(best_test_metrics_all)}'
)

log('')

log(
    'Test-oracle checkpoint saved only for diagnostics:'
)

log(
    str(
        RUN_ROOT
        / 'best_test_wga_oracle_diagnostic.pt'
    )
)


# ------------------------------------------------------------
# Output path
# ------------------------------------------------------------

log('')

log(
    f'Output directory: '
    f'{RUN_ROOT}'
)

log('=' * 100)


TRAINING FRESH FINAL CLASSIFIER WITH NEW FROZEN RATER
Architecture: Linear(2048 -> 2)
Calibration split: val_subset1, n=719
Selection split:   val_subset2, n=480
Test split:        test, n=5794

Normalization: global
Transform: sigmoid
Weight scale: 1.0
Correct weight: 1.0
Temperature: 1.0
Aux ordinary CE coefficient: 0.0

LR=0.01, momentum=0.9, weight_decay=0.0001
Weighted CE uses SUM and is NOT divided by sum(weights).

NOTE: highest test WGA across epochs will also be tracked as an ORACLE DIAGNOSTIC only.
The actual final model is still selected ONLY by the selection/validation metric.
[BEST TEST WGA OBSERVED] epoch=001, test_WGA=79.33%, selection_WGA=79.26%
[Epoch 001] online_cal_acc=81.08%, cal_eval_acc=90.40%, cal_WGA=83.45%, selection_acc=88.96%, selection_WGA=79.26%, test_acc=89.13%, test_WGA=79.33%, best_test_WGA_so_far=79.33%@001, mean_w=0.9194, mean_wrong_w=0.5738, wrong_seen=136
[Epoch 001 selection groups] g0=95.19%, g1=79.26%, g2=91.30%, g3=98.31%
[Epoch 001 test groups]

In [ ]:
# ============================================================
# 9. Final report on restored-best classifier
# ============================================================

for name, dataset in [
    ("calibration", calibration_dataset),
    ("selection", selection_dataset),
    ("test", test_dataset),
]:
    metrics = evaluate_classifier(
        model,
        dataset,
    )

    log("")
    log(f"[RESTORED BEST: {name.upper()}]")
    log(f"loss = {metrics['loss']:.6f}")
    log(f"acc  = {100.0 * metrics['accuracy']:.2f}%")
    log(f"WGA  = {100.0 * metrics['worst_group_accuracy']:.2f}%")
    log(format_group_acc(metrics))

print("\nDone. New frozen-Rater final-classifier run complete.")
print("Run directory:")
print(RUN_ROOT)
print("\nHistory CSV:")
print(HISTORY_PATH)
print("\nLog:")
print(LOG_PATH)



[RESTORED BEST: CALIBRATION]
loss = 0.197008
acc  = 99.58%
WGA  = 98.65%
g0=100.00%, g1=99.28%, g2=100.00%, g3=98.65%

[RESTORED BEST: SELECTION]
loss = 2.727464
acc  = 93.33%
WGA  = 89.13%
g0=94.65%, g1=94.15%, g2=89.13%, g3=89.83%

[RESTORED BEST: TEST]
loss = 2.384147
acc  = 94.56%
WGA  = 90.97%
g0=95.70%, g1=95.25%, g2=91.74%, g3=90.97%

Done. New frozen-Rater final-classifier run complete.
Run directory:
/content/drive/MyDrive/evidential_alignment_colab/frozen_new_rater_final_runs/waterbirds_erm_embeddings/new_frozen_rater_ermemb_global_sigmoid_x1p0_auxce_0p0_splitval_0p6_lr_0p01_20261005_083334

History CSV:
/content/drive/MyDrive/evidential_alignment_colab/frozen_new_rater_final_runs/waterbirds_erm_embeddings/new_frozen_rater_ermemb_global_sigmoid_x1p0_auxce_0p0_splitval_0p6_lr_0p01_20261005_083334/final_classifier_history.csv

Log:
/content/drive/MyDrive/evidential_alignment_colab/frozen_new_rater_final_runs/waterbirds_erm_embeddings/new_frozen_rater_ermemb_global_sigmoid_x1p0

## Preserved experiment structure

This updated notebook intentionally keeps the attached final-classifier
experiment unchanged except for the representation and frozen Rater.

### Splits

```python
calibration_split = "val_subset1"
selection_split   = "val_subset2"
test_split        = "test"
```

with:

```python
SPLIT_VAL = 0.6
```

### Final classifier

```python
Linear(2048, 2)
```

fresh random initialization, trained for 50 epochs with:

```python
SGD(
    lr=1e-2,
    momentum=0.9,
    weight_decay=1e-4,
)
```

### Active weighting configuration

```python
NORMALIZATION_MODE = "global"
WEIGHT_TRANSFORM = "sigmoid"
WEIGHT_SCALE = 1.0
CORRECT_WEIGHT = 1.0
TEMPERATURE = 1.0
AUX_CE_COEFF = 0.0
```

The Rater remains frozen throughout.

The notebook also retains the attached notebook's:

- epoch-wise actual-weight plots;
- wrong-only Rater-weight plots;
- history CSV;
- validation/WGA-based model selection;
- separate test-oracle diagnostic;
- best final-classifier checkpoint;
- restored-best final classifier checkpoint.